## **EMOTION Detection Model**



In [ ]:
# ── CELL 1: Upload Kaggle API Key ────────────────────────────
# Heading: Step 1: Upload Kaggle API Key

from google.colab import files
files.upload()

In [ ]:
# ── CELL 2: Setup Kaggle & Download FER-2013 ────────────────
# Heading: Step 2: Setup Kaggle & Download FER-2013 Dataset
# Go to: https://www.kaggle.com/datasets/msambare/fer2013
# Login → Download → you'll get fer2013.zip

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d fer2013

In [ ]:
# ── CELL 3: Check Folder Structure ──────────────────────────
# Heading: Step 3: Check Folder Structure
# Should look like:
# fer2013/
# ├── train/
# │   ├── angry/
# │   ├── disgusted/
# │   ├── fearful/
# │   ├── happy/
# │   ├── neutral/
# │   ├── sad/
# │   └── surprised/
# └── test/

!ls fer2013/train/
!ls fer2013/test/

In [ ]:

# ── CELL 4: Import Libraries ─────────────────────────────────
# Heading: 1. Install Required Libraries & Import Modules

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os

print("TensorFlow version:", tf.__version__)

In [ ]:
# ── CELL 5: Prepare the Data ─────────────────────────────────
# Heading: 4. Prepare the Data

IMG_SIZE = 100          # Sir's PDF uses 100x100
BATCH_SIZE = 32

train_dir = 'fer2013/train'
val_dir   = 'fer2013/test'

train_gen = ImageDataGenerator(rescale=1./255)
val_gen   = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'       # Sir uses categorical (multi-class)
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Class labels:", train_data.class_indices)

In [ ]:
# ── CELL 6: Build the CNN Model ──────────────────────────────
# Heading: 5. Build the CNN Model

model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
    # train_data.num_classes = 7 for FER-2013
])

In [ ]:
# ── CELL 7: Compile the Model ────────────────────────────────
# Heading: 6. Compile the Model

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── CELL 8: Train the Model ──────────────────────────────────
# Heading: 7. Train the Model

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5a
)

In [ ]:
# ── CELL 9: Evaluate and Plot ────────────────────────────────
# Heading: 8. Evaluate and Plot

# Evaluate
test_loss, test_acc = model.evaluate(val_data)
print(f'Test Accuracy: {test_acc:.2f}')

# Plot training history
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.show()

In [ ]:
# ── CELL 10: Upload a Face Image to Test ─────────────────────
# Heading: Step 9: Upload a Face Image to Test

from google.colab import files
uploaded = files.upload()

In [ ]:
# ── CELL 11: Predict Emotion ─────────────────────────────────
# Heading: Step 10: Predict Emotion on Uploaded Image

import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

image_path = list(uploaded.keys())[0]
img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
plt.imshow(img)
plt.axis('off')
plt.title('Uploaded Image')
plt.show()

img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array /= 255.0

pred = model.predict(img_array)
pred_class_index = np.argmax(pred)

# Access the correct attribute 'class_indices'
pred_class_label = list(train_data.class_indices.keys())[pred_class_index]
print("Predicted class =", pred_class_label)